In [1]:
from os import chdir
from pathlib import Path

cwd = Path.cwd()
print(f"CWD: {cwd}")
if cwd.name == "code":
    chdir("..")
print(f"CWD: {Path.cwd()}")

CWD: /home/james/git/research/fed-learning/code
CWD: /home/james/git/research/fed-learning


In [15]:
from src.models.GeneralizedMF import GeneralizedMF, GeneralizedMFOneLayer, GeneralizedMFSharedItemLayer
from src.models.MatrixFactorization import MF
from src.graphs import random_k_out_graph
from src.users import User
from src.training.decentralized import decentralized_train_loop, decentralized_validate_loop, share_gradient

In [3]:
#Make data sample iterable
from torch.utils.data import Dataset, DataLoader, TensorDataset, Sampler
import torch
from tqdm.notebook import tqdm

import pandas as pd
import numpy as np

import torch.nn as nn
import torch.nn.functional as F
from torch.optim import SGD
from torch.optim.lr_scheduler import ReduceLROnPlateau

from collections import Counter
import networkx as nx
from networkx.generators.classic import empty_graph
from networkx.utils import discrete_sequence, py_random_state, weighted_choice

In [16]:
gen = GeneralizedMF(2, n_factors=3)
mf = MF(2, 3, 3)
gmf_shared = GeneralizedMFSharedItemLayer(2, 3)

In [17]:
list(gmf_shared.named_parameters())

[('user_factors.weight',
  Parameter containing:
  tensor([[ 0.1499, -0.4609, -0.6716]], requires_grad=True)),
 ('item_factors.weight',
  Parameter containing:
  tensor([[ 1.4475,  0.1206,  0.6822],
          [-1.3094, -0.4588, -0.4868]], requires_grad=True)),
 ('user_bias.weight',
  Parameter containing:
  tensor([[-0.2611]], requires_grad=True)),
 ('item_bias.weight',
  Parameter containing:
  tensor([[-1.5372],
          [-1.8276]], requires_grad=True)),
 ('user_layers.weight',
  Parameter containing:
  tensor([[-0.0564,  0.5219,  0.2035],
          [ 0.2550,  0.5087,  0.0935],
          [ 0.4275, -0.5289, -0.1570],
          [ 0.1621,  0.3110,  0.4261],
          [-0.1476, -0.3598, -0.3054],
          [-0.4951,  0.3168,  0.3084],
          [-0.4126, -0.0424, -0.3367],
          [-0.2102,  0.2116,  0.2769],
          [-0.2809, -0.1601,  0.2925],
          [ 0.5379, -0.1488,  0.0157]], requires_grad=True)),
 ('user_layers.bias',
  Parameter containing:
  tensor([ 0.0215,  0.3102, -0.

In [19]:
gen.get_parameter("item_factors.weight").grad

In [8]:
for name, param in mf.named_parameters():
    if "item" in name:
        print(name)
        print(param.grad)

item_factors.weight
None
item_bias.weight
None


In [ ]:
mf.

In [14]:
shared_param_names = ["item_factors.weight", "item_bias.weight", "item_layer_dict.item0.weight"]
for name, param in gen.named_parameters():
    if "item" in name:
        print(name)
        print(param.grad)

item_factors.weight
None
item_bias.weight
None
item_layer_dict.item0.weight
None
item_layer_dict.item0.bias
None
item_layer_dict.item1.weight
None
item_layer_dict.item1.bias
None


In [7]:
for name in shared_param_names:
    p = mf.get_parameter(name)
    print(p)
    print(p.grad)

Parameter containing:
tensor([[ 0.2013,  0.2735, -0.5667],
        [ 1.1590,  0.2281,  2.2081],
        [ 1.0338,  1.3150,  0.1004]], requires_grad=True)
None
Parameter containing:
tensor([[-1.3817],
        [ 1.7311],
        [ 0.8881]], requires_grad=True)
None


AttributeError: MF has no attribute `item_layer_dict`

In [22]:
for name in shared_param_names:
    p = gen.get_parameter(name)
    print(p)
    print(p.grad)

Parameter containing:
tensor([[-0.2693, -0.7266,  1.1907],
        [ 0.0955,  0.0503, -0.3212]], requires_grad=True)
None
Parameter containing:
tensor([[1.7796],
        [0.0685]], requires_grad=True)
None
Parameter containing:
tensor([[-0.2930, -0.1689, -0.1985],
        [ 0.0935, -0.0335,  0.2945],
        [ 0.2024, -0.2425,  0.3148],
        [ 0.3046, -0.5512, -0.0620],
        [-0.2486,  0.1924,  0.0023],
        [ 0.4455, -0.1096,  0.5186],
        [ 0.0480,  0.4407,  0.0259],
        [-0.5750,  0.1558, -0.0806],
        [ 0.1093,  0.0530,  0.1105],
        [-0.1769,  0.3117, -0.5525]], requires_grad=True)
None


In [ ]:
share_gradient(0,)

In [4]:
train_df = pd.read_csv("dataset/hm_train_df.csv")
val_df = pd.read_csv("dataset/hm_val_df.csv")
n_users = train_df['customer_id'].nunique()
n_items = train_df['product_code'].nunique()
X_train = train_df[["customer_id", "product_code"]]
X_train.columns = ["user_id", "item_id"]
y_train = train_df["bought"]

X_test = val_df[["customer_id", "product_code"]]
X_test.columns = ["user_id", "item_id"]
y_test = val_df["bought"]
# y_train.columns = [""]

In [5]:
users = {}
for i in tqdm(range(n_users)):
    # model = MF(n_users=n_users, n_items=n_items)
    model = GeneralizedMFOneLayer(n_users=n_users, n_items=n_items)
    users[i] = User(name=i, model=model, optimizer=SGD(model.parameters(), lr=.01))

  0%|          | 0/1760 [00:00<?, ?it/s]

In [6]:
graph = random_k_out_graph(n_users,5,50)

In [7]:
val_dataset = TensorDataset(torch.tensor(X_test.to_numpy(), dtype=int), torch.tensor(y_test.to_numpy(), dtype=float))
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=True)
train_tensor_dataset = TensorDataset(torch.tensor(X_train.to_numpy()), torch.tensor(y_train.to_numpy()))
train_tensor_loader = DataLoader(train_tensor_dataset, batch_size=1, shuffle=True)

In [8]:
for idx, (inputs, targets) in enumerate(train_tensor_loader):
    print(inputs.shape, inputs)
    print(targets.shape, targets)
    break

torch.Size([1, 2]) tensor([[ 385, 1777]])
torch.Size([1]) tensor([1])


In [9]:
val_losses = new_train(users, train_tensor_loader, val_loader, epochs=5, graph=graph)

/home/james/git/research/fed-learning/.venv/lib/python3.10/site-packages/torch/nn/modules/loss.py:535: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
Epoch 1 Loss: nan :   5%|████▍                                                                                          | 4073/86480 [00:14<04:50, 283.98it/s]


KeyboardInterrupt: 

In [1]:
from torch import Tensor, IntTensor
from torch.nn import Embedding

In [2]:
e = Embedding(3, 3)

In [3]:
e(IntTensor([0]))

tensor([[-0.3655, -0.4683, -0.7775]], grad_fn=<EmbeddingBackward0>)

In [4]:
Tensor([1, 2, 3 ]) + e(IntTensor([0]))

tensor([[0.6345, 1.5317, 2.2225]], grad_fn=<AddBackward0>)

In [7]:
e = Embedding(1, 10)

In [11]:
e(IntTensor([1]))

IndexError: index out of range in self